# Step 20 — Individual-event phase-file indexing

The workflow is implemented directly in this notebook. Shared parsing/filter helpers remain in `pinatubo_steps.pha_parser`; step 20 also uses FLOVOpy's VDAP reader. This removes the redundant step-specific module while retaining a single shared implementation of the underlying parsing rules.

Legacy PHA files are read without modification. Running the processing cell replaces the configured pick-index CSV and error log beneath the external output root. These are arrival picks and source groupings, not relocated earthquakes or confirmed waveform associations. No clock offset/drift correction is applied. UTC-style timestamps reflect parser representation, not independently verified historical timing.

Run both steps 20 and 21 before step 22, which applies the cross-source merge/priority rules. Review error logs and grouping assumptions before merging.

## 1. Load and review configuration

Choose `CONFIG_FILE` if needed. Inspect source and destination paths and filter settings before executing the processing cell. This cell only resolves settings.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    pha_root=P['LEGACY_PHA_INDIVIDUAL_DIR'],
    out_csv=P['INDIV_PICK_INDEX'],
    error_log=P['INDIV_LOGFILE'],
    glob='**/*.PHA',
)
display(SETTINGS)

## 2. Reader and workflow implementation

Each input filename stem supplies `event_id`; it is a legacy grouping key, not proof of a unique physical earthquake. Individual-file picks receive preferred status later in the merge workflow; they are not intrinsically error-free.

The FLOVOpy reader runs in strict mode with network `XB`. The explicitly enabled terminal-separator exception permits a single recognized final separator (10/100) after picks, with only blank/EOF material following it; malformed internal records still fail. Deduplication keeps the first pick for a SEED ID and phase within 0.001 seconds. The local outlier rule keeps picks within ±60 seconds of the median when at least three picks are available; this is not a 60-second maximum total event duration.

Rows retain original line text/number, source filename and full path, phase, timing, mapped ID, onset, polarity and weight. Pick IDs use a running counter and are not permanent identifiers across changed input inventories. The filename glob defaults to recursive uppercase `*.PHA`; change it deliberately if your archive uses another convention.

A parsing exception is logged and raised immediately. The final CSV is written only after the whole loop succeeds; an older CSV may remain if a run fails, so do not mistake it for a successful new run. Empty groups are skipped; an entirely empty result stops the workflow.

The next cell defines functions only. It does not read data or write products.

In [ ]:
from pathlib import Path

import pandas as pd

from flovopy.core.vdap import read_vdap_pha_file

from pinatubo_steps.pha_parser import (
    _dedupe_picks,
    filter_pick_outliers,
)

def extract_event_id(pha_path: Path) -> str:
    return pha_path.stem

def run(
    *,
    pha_root,
    out_csv,
    error_log,
    glob='**/*.PHA'
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        pha_root=pha_root,
        out_csv=out_csv,
        error_log=error_log,
        glob=glob,
    )

    pha_root = Path(args.pha_root)
    out_csv = Path(args.out_csv)
    error_log = Path(args.error_log)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    error_log.parent.mkdir(parents=True, exist_ok=True)

    pha_files = sorted(pha_root.glob(args.glob))
    if not pha_files:
        raise SystemExit(f"No PHA files found under {pha_root}")

    print(f"Found {len(pha_files)} individual PHA files")

    rows = []
    errors = []
    pick_counter = 0

    for pha_file in pha_files:
        event_id = extract_event_id(pha_file)

        try:
            picks = read_vdap_pha_file(pha_file, network="XB", strict=True, allow_terminal_separator=True)
            picks = _dedupe_picks(picks)
            picks = filter_pick_outliers(picks, max_span_seconds=60)
        except Exception as e:
            errors.append(f"{pha_file}: {e}")
            error_log.write_text("\n".join(errors) + "\n")
            raise  # FAIL LOUDLY

        if not picks:
            continue

        for p in picks:
            pick_counter += 1
            rows.append({
                "raw_line": p.get("raw_line"),
                "raw_lineno": p.get("raw_lineno"),
                "event_id": event_id,
                "event_id_source": "individual_pha_filename",
                "pick_id": f"indpha_{event_id}_{pick_counter}",
                "pha_file": pha_file.name,
                "pha_path": str(pha_file.resolve()),
                "station": p.get("station"),
                "channel": p.get("channel"),
                "seed_id": p.get("seed_id"),
                "phase": p.get("phase"),
                "pick_time": str(p.get("time")),
                "onset": p.get("onset"),
                "first_motion": p.get("first_motion"),
                "weight": p.get("weight"),
            })

    if not rows:
        raise SystemExit("STEP 20 FAILED: no picks parsed")

    df = (
        pd.DataFrame(rows)
        .sort_values(["event_id", "station", "phase", "pick_time"])
        .reset_index(drop=True)
    )

    df.to_csv(out_csv, index=False)

    with open(error_log, "w") as f:
        for line in errors:
            f.write(line + "\n")

    print("\n=== STEP 20 SUMMARY ===")
    print(f"Legacy events: {df['event_id'].nunique()}")
    print(f"Total picks:   {len(df)}")
    print(f"CSV written:   {out_csv}")
    print(f"Errors logged: {error_log}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Run the step

This cell performs parsing and writes the configured CSV and log. Inputs remain unchanged. Review the printed summary and any exceptions before continuing.

In [ ]:
result = run(**SETTINGS)

## 4. Inspect results and error log

Preview the pick table and recorded issues. For monthly sources, pay particular attention to trimmed/dropped blocks. Then run the other phase-indexing notebook and continue to step 22; do not infer waveform matches from these source IDs alone.

In [ ]:
display(result['df'].head())
print('Output rows:', len(result['df']))
print('Source groups:', result['df']['event_id'].nunique())
log = Path(SETTINGS['error_log'])
print('Error log:', log)
print(log.read_text() if log.exists() and log.stat().st_size else 'No logged issues in this completed run.')